Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\24pasos_lstm_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [6]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [7]:
futuros = 24
pasados  = 12

In [8]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [9]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 7)
Dimensiones de Y: (43765, 1)


In [10]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777
          nan]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536
          nan]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295
          nan]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181
          nan]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894
          nan]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698
          nan]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584
          nan]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347
          nan]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356
          nan]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241
          nan]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449
          nan]
 [ 0.57004095 -0.656257

Se dividen nuevamente los conjuntos de datos

In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 12, 7)
Las dimensiones de testX son:  (8797, 12, 7)
Las dimensiones de valX son:  (4333, 12, 7)


In [12]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [13]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [14]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [15]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [16]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

193/193 - 14s - 70ms/step - ia: 0.1681 - loss: 1.2039 - mae: 0.8290 - rmse: 1.0748 - smape: 1.7170 - val_ia: 0.2591 - val_loss: 0.5384 - val_mae: 0.5674 - val_rmse: 0.6542 - val_smape: 1.6185

Epoch 2/128                                           

193/193 - 3s - 16ms/step - ia: 0.2346 - loss: 1.0681 - mae: 0.7768 - rmse: 1.0126 - smape: 1.5558 - val_ia: 0.2557 - val_loss: 0.5015 - val_mae: 0.5234 - val_rmse: 0.6155 - val_smape: 1.3063

Epoch 3/128                                           

193/193 - 3s - 17ms/step - ia: 0.2920 - loss: 1.0406 - mae: 0.7567 - rmse: 1.0007 - smape: 1.4156 - val_ia: 0.2571 - val_loss: 0.4981 - val_mae: 0.5145 - val_rmse: 0.6098 - val_smape: 1.2629

Epoch 4/128                                           

193/193 - 4s - 19ms/step - ia: 0.3202 - loss: 1.0145 - mae: 0.7462 - rmse: 0.9898 - smape: 1.3696 - val_ia: 0.2585 - val_loss: 0.4952 - val_mae: 0.5125 - val_rmse: 0.6094 - val_smape: 1.2577

Epoch 5

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                         

25/25 - 16s - 652ms/step - ia: 0.1318 - loss: 1.0934 - mae: 0.7757 - rmse: 1.0258 - smape: 1.7191 - val_ia: 0.2377 - val_loss: 0.4779 - val_mae: 0.5044 - val_rmse: 0.6770 - val_smape: 1.2307

Epoch 2/16                                                                         

25/25 - 4s - 142ms/step - ia: 0.3511 - loss: 0.9734 - mae: 0.7295 - rmse: 0.9652 - smape: 1.3029 - val_ia: 0.2757 - val_loss: 0.4755 - val_mae: 0.4936 - val_rmse: 0.6743 - val_smape: 1.1869

Epoch 3/16                                                                         

25/25 - 5s - 199ms/step - ia: 0.4149 - loss: 0.8861 - mae: 0.6970 - rmse: 0.9492 - smape: 1.2220 - val_ia: 0.2593 - val_loss: 0.5092 - val_mae: 0.5320 - val_rmse: 0.6957 - val_smape: 1.2914

Epoch 4/16                                                                         

25/25 - 3s - 136ms/step - ia: 0.4056 - loss: 0.9049 - mae: 0.7060 - rmse: 0.9518 - sma

...

Epoch 1/8                                                                        

193/193 - 18s - 92ms/step - ia: 0.3802 - loss: 0.9725 - mae: 0.7102 - rmse: 0.9591 - smape: 1.2579 - val_ia: 0.2550 - val_loss: 0.4631 - val_mae: 0.5111 - val_rmse: 0.6067 - val_smape: 1.2470

Epoch 2/8                                                                        

193/193 - 3s - 15ms/step - ia: 0.4087 - loss: 0.9054 - mae: 0.6947 - rmse: 0.9307 - smape: 1.1968 - val_ia: 0.2588 - val_loss: 0.4852 - val_mae: 0.5172 - val_rmse: 0.6090 - val_smape: 1.2391

Epoch 3/8                                                                        

193/193 - 3s - 16ms/step - ia: 0.4611 - loss: 0.8273 - mae: 0.6556 - rmse: 0.8953 - smape: 1.1283 - val_ia: 0.2461 - val_loss: 1.0554 - val_mae: 0.6167 - val_rmse: 0.7300 - val_smape: 1.2197

Epoch 4/8                                                                        

193/193 - 3s - 17ms/step - ia: 0.4655 - loss: 0.8251 - mae: 0.6534 - rmse: 0.8895 - smape: 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



97/97 - 8s - 80ms/step - ia: 0.1806 - loss: 1.1135 - mae: 0.7875 - rmse: 1.0484 - smape: 1.6200 - val_ia: 0.2476 - val_loss: 0.5196 - val_mae: 0.5599 - val_rmse: 0.6777 - val_smape: 1.6212

Epoch 2/256                                                                      

97/97 - 2s - 18ms/step - ia: 0.1991 - loss: 1.0823 - mae: 0.7782 - rmse: 1.0301 - smape: 1.5886 - val_ia: 0.2531 - val_loss: 0.5029 - val_mae: 0.5462 - val_rmse: 0.6653 - val_smape: 1.5334

Epoch 3/256                                                                      

97/97 - 2s - 18ms/step - ia: 0.2279 - loss: 1.0498 - mae: 0.7647 - rmse: 1.0108 - smape: 1.5335 - val_ia: 0.2572 - val_loss: 0.4900 - val_mae: 0.5350 - val_rmse: 0.6553 - val_smape: 1.4604

Epoch 4/256                                                                      

97/97 - 2s - 18ms/step - ia: 0.2526 - loss: 1.0188 - mae: 0.7564 - rmse: 1.0025 - smape: 1.5013 - val_ia: 0.2593 - val_loss: 0.4805 - val_mae: 0.5265 - val_rmse: 0.6476 - val_smape:

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                       

770/770 - 25s - 33ms/step - ia: 0.3350 - loss: 1.0365 - mae: 0.7396 - rmse: 0.9436 - smape: 1.3710 - val_ia: 0.2204 - val_loss: 0.5120 - val_mae: 0.5177 - val_rmse: 0.5601 - val_smape: 1.2618

Epoch 2/16                                                                       

770/770 - 11s - 15ms/step - ia: 0.3881 - loss: 0.8879 - mae: 0.6903 - rmse: 0.8847 - smape: 1.2383 - val_ia: 0.2097 - val_loss: 0.5589 - val_mae: 0.5185 - val_rmse: 0.5613 - val_smape: 1.1527

Epoch 3/16                                                                       

770/770 - 11s - 14ms/step - ia: 0.4077 - loss: 0.8575 - mae: 0.6770 - rmse: 0.8681 - smape: 1.2074 - val_ia: 0.2173 - val_loss: 0.4942 - val_mae: 0.5068 - val_rmse: 0.5483 - val_smape: 1.2336

Epoch 4/16                                                                       

770/770 - 11s - 14ms/step - ia: 0.4282 - loss: 0.8093 - mae: 0.6594 - rmse: 0.8419 - smap

In [18]:
print(best)

{'activation': 1, 'batch': 3, 'dropout': 0.2, 'epochs': 1, 'layers': 1.0, 'learning_rate': 0.002508532347133338, 'units': 4}
